# 117. Voronoi分割: all-MiniLM-L6-v2 の評価とセントロイドエクスポート

## 目的
- all-MiniLM-L6-v2 (384D, 英語) でVoronoi分割を評価
- E5-base (768D) との比較
- セントロイドをNumPy + JSON形式でエクスポート

## モデル特性
| | all-MiniLM-L6-v2 | multilingual-e5-base |
|---|---|---|
| 次元 | 384 | 768 |
| 言語 | 英語 | 多言語 |
| プレフィックス | **不要** | query:/passage: |
| パラメータ | 22M | 278M |

## NB114参考値 (E5-base EN, C=256)
| assign | P | R@10 | 候補% |
|--------|---|------|-------|
| 2 | 2 | 78.9% | 2.8% |
| 2 | 5 | 88.5% | 6.5% |

In [1]:
import numpy as np
import json
from pathlib import Path
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')
np.random.seed(42)

N_QUERIES = 200
TOP_K = 10

## 1. データロードと空間分析

In [2]:
emb = np.load(DATA_DIR / '10k_minilm_embeddings.npy').astype(np.float32)
print(f'MiniLM: {emb.shape}, dtype={emb.dtype}')

norms = np.linalg.norm(emb, axis=1, keepdims=True)
emb_normed = emb / norms
print(f'Norm: mean={np.linalg.norm(emb_normed, axis=1).mean():.6f}')

# 空間分析（NB113と同じ）
rng = np.random.default_rng(42)
idx1 = rng.choice(len(emb), 5000, replace=True)
idx2 = rng.choice(len(emb), 5000, replace=True)
mask = idx1 != idx2
cos_rand = np.sum(emb_normed[idx1[mask]] * emb_normed[idx2[mask]], axis=1)

query_ids = rng.choice(len(emb), 200, replace=False)
knn_sims = []
for qi in query_ids:
    sims = emb_normed[qi] @ emb_normed.T
    sims[qi] = -1
    knn_sims.append(np.mean(np.sort(sims)[-10:]))

print(f'\n=== Embedding空間分析 ===')
print(f'  ランダムペア cos: mean={cos_rand.mean():.4f}, std={cos_rand.std():.4f}')
print(f'  k=10近傍 cos:     mean={np.mean(knn_sims):.4f}, std={np.std(knn_sims):.4f}')
print(f'  Gap:              {np.mean(knn_sims) - cos_rand.mean():.4f}')
print(f'\n--- NB113参考値 ---')
print(f'  E5-base EN: ランダム=0.706, 近傍=0.815, Gap=0.109')
print(f'  E5-base JA: ランダム=0.765, 近傍=0.873, Gap=0.108')

MiniLM: (10000, 384), dtype=float32
Norm: mean=1.000000

=== Embedding空間分析 ===
  ランダムペア cos: mean=0.0195, std=0.0882
  k=10近傍 cos:     mean=0.4400, std=0.0896
  Gap:              0.4205

--- NB113参考値 ---
  E5-base EN: ランダム=0.706, 近傍=0.815, Gap=0.109
  E5-base JA: ランダム=0.765, 近傍=0.873, Gap=0.108


## 2. k-means構築とグリッドサーチ (assign=1,2 × C × P)

In [3]:
def precompute_ground_truth(embeddings, query_indices, top_k=10):
    gt_dict = {}
    cos_all = cosine_similarity(embeddings[query_indices], embeddings)
    for i, qi in enumerate(query_indices):
        cos_all[i, qi] = -1
        gt_dict[qi] = set(np.argsort(cos_all[i])[-top_k:])
    return gt_dict


def build_multi_assign(all_sims, n_assign):
    N, C = all_sims.shape
    partitions = {c: [] for c in range(C)}
    for i in range(N):
        for pid in np.argsort(-all_sims[i])[:n_assign]:
            partitions[pid].append(i)
    for c in range(C):
        partitions[c] = np.array(partitions[c], dtype=int)
    return partitions


def evaluate(emb_normed, centroids, partitions, gt_dict, qi, n_probes, emb, top_k=10):
    recalls, cands_list = [], []
    for q in qi:
        gt = gt_dict[q]
        top_c = np.argsort(-(centroids @ emb_normed[q]))[:n_probes]
        cands = set()
        for c in top_c:
            cands.update(partitions[c].tolist())
        cands.discard(q)
        cands = np.array(list(cands))
        cands_list.append(len(cands))
        if len(cands) > 0:
            s = cosine_similarity(emb[q:q+1], emb[cands])[0]
            top_in = cands[np.argsort(-s)[:top_k]]
            recalls.append(len(gt & set(top_in)) / top_k)
        else:
            recalls.append(0.0)
    return np.mean(recalls), np.mean(cands_list)


# k-meansモデル構築
cluster_range = [32, 64, 128, 256]
kmeans_models = {}

for n_c in cluster_range:
    km = MiniBatchKMeans(n_clusters=n_c, random_state=42, batch_size=2048, n_init=3)
    km.fit(emb_normed)
    c = km.cluster_centers_
    c_normed = c / np.linalg.norm(c, axis=1, keepdims=True)
    all_sims = emb_normed @ c_normed.T
    kmeans_models[n_c] = {'centroids': c_normed, 'all_sims': all_sims}
    sizes = [np.sum(km.labels_ == i) for i in range(n_c)]
    print(f'C={n_c}: mean={np.mean(sizes):.1f}, min={np.min(sizes)}, '
          f'max={np.max(sizes)}, CV={np.std(sizes)/np.mean(sizes):.3f}')

# GT事前計算
qi = rng.choice(len(emb), N_QUERIES, replace=False)
gt = precompute_ground_truth(emb, qi)
print(f'\n{N_QUERIES} queries, GT precomputed')

# グリッドサーチ
probe_range = [1, 2, 3, 4, 5, 8, 10, 15, 20]
assign_range = [1, 2, 3]
results = []

for n_c in cluster_range:
    m = kmeans_models[n_c]
    for n_a in assign_range:
        parts = build_multi_assign(m['all_sims'], n_a)
        for n_p in probe_range:
            if n_p > n_c:
                continue
            r, c = evaluate(emb_normed, m['centroids'], parts, gt, qi, n_p, emb)
            results.append({
                'n_clusters': n_c, 'n_assign': n_a, 'n_probes': n_p,
                'recall': r, 'candidates': c, 'cand_ratio': c / len(emb),
            })

print(f'Total configs: {len(results)}')

C=32: mean=312.5, min=134, max=824, CV=0.438


C=64: mean=156.2, min=42, max=357, CV=0.386


C=128: mean=78.1, min=3, max=190, CV=0.510


C=256: mean=39.1, min=1, max=159, CV=0.764

200 queries, GT precomputed


Total configs: 108


## 3. C=256 結果テーブル と E5-base比較

In [4]:
# C=256 の全assign×probe結果
print('='*70)
print('C=256 結果テーブル')
print('='*70)

for n_a in assign_range:
    print(f'\n--- assign={n_a} ---')
    print(f'{"P":>4} {"R@10":>8} {"候補数":>8} {"候補%":>8}')
    print('-' * 32)
    for r in sorted([x for x in results if x['n_clusters']==256 and x['n_assign']==n_a],
                    key=lambda x: x['n_probes']):
        print(f'{r["n_probes"]:>4} {r["recall"]*100:>7.1f}% '
              f'{r["candidates"]:>7.0f} {r["cand_ratio"]*100:>7.1f}%')

# E5-base EN (NB114) との比較
print('\n' + '='*70)
print('E5-base EN (NB114) vs MiniLM 比較 (C=256, assign=2)')
print('='*70)

e5_ref = {
    2: (78.9, 2.8), 3: (83.8, 4.0), 5: (88.5, 6.5),
    8: (92.4, 10.1), 10: (94.2, 12.4),
}

print(f'{"P":>4} {"E5 R@10":>10} {"E5 候補%":>10} {"MiniLM R@10":>12} {"MiniLM 候補%":>13} {"差":>8}')
print('-' * 58)
for n_p in [2, 3, 5, 8, 10]:
    mini = [x for x in results if x['n_clusters']==256 and x['n_assign']==2 and x['n_probes']==n_p]
    if mini and n_p in e5_ref:
        m = mini[0]
        e5_r, e5_c = e5_ref[n_p]
        diff = m['recall']*100 - e5_r
        print(f'{n_p:>4} {e5_r:>9.1f}% {e5_c:>9.1f}% '
              f'{m["recall"]*100:>11.1f}% {m["cand_ratio"]*100:>12.1f}% {diff:>+7.1f}pp')

C=256 結果テーブル

--- assign=1 ---
   P     R@10      候補数      候補%
--------------------------------
   1    50.5%      59     0.6%
   2    68.1%     110     1.1%
   3    75.5%     161     1.6%
   4    79.7%     209     2.1%
   5    82.3%     255     2.5%
   8    87.7%     386     3.9%
  10    89.7%     475     4.7%
  15    92.6%     683     6.8%
  20    94.5%     884     8.8%

--- assign=2 ---
   P     R@10      候補数      候補%
--------------------------------
   1    65.0%     111     1.1%
   2    79.6%     196     2.0%
   3    84.7%     277     2.8%
   4    87.7%     355     3.5%
   5    89.6%     427     4.3%
   8    93.1%     630     6.3%
  10    94.5%     765     7.6%
  15    96.5%    1093    10.9%
  20    97.5%    1396    14.0%

--- assign=3 ---
   P     R@10      候補数      候補%
--------------------------------
   1    73.0%     160     1.6%
   2    84.8%     275     2.8%
   3    88.9%     382     3.8%
   4    91.3%     485     4.9%
   5    92.6%     581     5.8%
   8    95.3%     846    

## 4. Recall目標別の推奨構成

In [5]:
print('='*70)
print('Recall目標別 最小コスト構成')
print('='*70)

for target in [0.95, 0.90, 0.85, 0.80, 0.75]:
    candidates = [r for r in results if r['recall'] >= target]
    if not candidates:
        print(f'  R@10≥{target*100:.0f}%: 達成する構成なし')
        continue
    best = min(candidates, key=lambda x: x['candidates'])
    config = f'C={best["n_clusters"]},A={best["n_assign"]},P={best["n_probes"]}'
    print(f'  R@10≥{target*100:.0f}%: {config:<22} R@10={best["recall"]*100:.1f}% '
          f'候補={best["candidates"]:.0f} ({best["cand_ratio"]*100:.1f}%) IN句={best["n_probes"]}')

Recall目標別 最小コスト構成
  R@10≥95%: C=256,A=3,P=8          R@10=95.3% 候補=846 (8.5%) IN句=8
  R@10≥90%: C=256,A=3,P=4          R@10=91.3% 候補=485 (4.9%) IN句=4
  R@10≥85%: C=256,A=2,P=4          R@10=87.7% 候補=355 (3.5%) IN句=4
  R@10≥80%: C=256,A=1,P=5          R@10=82.3% 候補=255 (2.5%) IN句=5
  R@10≥75%: C=256,A=1,P=3          R@10=75.5% 候補=161 (1.6%) IN句=3


## 5. セントロイドエクスポート (NumPy + JSON)

In [6]:
# C=256のセントロイドをエクスポート
centroids_256 = kmeans_models[256]['centroids']

# NumPy
npy_path = DATA_DIR / 'voronoi_centroids_256_minilm.npy'
np.save(npy_path, centroids_256)
loaded = np.load(npy_path)
assert np.allclose(loaded, centroids_256)
print(f'NumPy: {npy_path.name} ({npy_path.stat().st_size/1024:.1f} KB)')
print(f'  Shape: {loaded.shape}, dtype: {loaded.dtype}')

# JSON
json_path = DATA_DIR / 'voronoi_centroids_256_minilm.json'

# 推奨構成をresultsから算出
rec = {}
for target, label in [(0.90, 'high_recall'), (0.85, 'balanced'), (0.80, 'low_cost')]:
    cands = [r for r in results if r['recall'] >= target]
    if cands:
        best = min(cands, key=lambda x: x['candidates'])
        rec[label] = {
            'assign': best['n_assign'], 'probes': best['n_probes'],
            'note': f'R@10>={target*100:.0f}%',
            'recall': round(best['recall'] * 100, 1),
            'candidate_ratio': round(best['cand_ratio'] * 100, 1),
        }

export_data = {
    'metadata': {
        'model': 'sentence-transformers/all-MiniLM-L6-v2',
        'dimension': int(centroids_256.shape[1]),
        'n_clusters': int(centroids_256.shape[0]),
        'training_data': 'Wikipedia 10K EN (English only)',
        'training_samples': int(len(emb)),
        'normalized': True,
        'prefix_required': False,
        'kmeans_params': {
            'random_state': 42,
            'batch_size': 2048,
            'n_init': 3,
        },
        'recommended_configs': rec,
        'usage': {
            'zope': 'KeywordIndex pivot_ids, query with operator="or"',
            'firestore': 'array field pivot_ids, query with array-contains-any',
        },
    },
    'centroids': centroids_256.tolist(),
}

with open(json_path, 'w') as f:
    json.dump(export_data, f, ensure_ascii=False)

print(f'JSON:  {json_path.name} ({json_path.stat().st_size/1024:.1f} KB)')

# 検証
with open(json_path) as f:
    d = json.load(f)
assert np.array(d['centroids']).shape == centroids_256.shape
print(f'\nMetadata:')
for k, v in d['metadata'].items():
    print(f'  {k}: {v}')

NumPy: voronoi_centroids_256_minilm.npy (384.1 KB)
  Shape: (256, 384), dtype: float32
JSON:  voronoi_centroids_256_minilm.json (2109.8 KB)

Metadata:
  model: sentence-transformers/all-MiniLM-L6-v2
  dimension: 384
  n_clusters: 256
  training_data: Wikipedia 10K EN (English only)
  training_samples: 10000
  normalized: True
  prefix_required: False
  kmeans_params: {'random_state': 42, 'batch_size': 2048, 'n_init': 3}
  recommended_configs: {'high_recall': {'assign': 3, 'probes': 4, 'note': 'R@10>=90%', 'recall': 91.3, 'candidate_ratio': 4.9}, 'balanced': {'assign': 2, 'probes': 4, 'note': 'R@10>=85%', 'recall': 87.6, 'candidate_ratio': 3.5}, 'low_cost': {'assign': 1, 'probes': 5, 'note': 'R@10>=80%', 'recall': 82.3, 'candidate_ratio': 2.5}}
  usage: {'zope': 'KeywordIndex pivot_ids, query with operator="or"', 'firestore': 'array field pivot_ids, query with array-contains-any'}


## 6. 評価・考察

### MiniLMはE5-baseより少ない候補数で同等以上のRecallを達成

C=256, assign=2での直接比較:

| P | E5 R@10 | E5 候補% | MiniLM R@10 | MiniLM 候補% | Recall差 |
|---|---------|---------|------------|-------------|---------|
| 2 | 78.9% | 2.8% | **79.6%** | **2.0%** | +0.7pp |
| 5 | 88.5% | 6.5% | **89.6%** | **4.3%** | +1.1pp |
| 10 | 94.2% | 12.4% | **94.5%** | **7.6%** | +0.2pp |

MiniLMは**同じRecallをE5の約60-70%の候補数**で達成。

### 原因: MiniLMのランダム-近傍Gapが画像に近い

| モデル | ランダムcos | 近傍cos | Gap |
|--------|-----------|---------|-----|
| 画像(顔認識) | ~0.05 | ~0.60 | **~0.55** |
| **MiniLM** | **0.020** | **0.440** | **0.421** |
| E5-base EN | 0.706 | 0.815 | 0.109 |
| E5-base JA | 0.765 | 0.873 | 0.108 |

MiniLMのGap(0.42)はE5(0.11)の**3.8倍**で、画像(0.55)に近い。
これはMiniLMの384D空間がE5の768D空間よりコンパクトで、近傍と非近傍の分離が良いことを意味する。

### 推奨構成

| 目標 | 構成 | R@10 | 候補% | IN句 |
|------|------|------|-------|------|
| R@10≥95% | C=256, A=3, P=8 | 95.3% | 8.5% | 8 |
| R@10≥90% | C=256, A=3, P=4 | 91.3% | 4.9% | 4 |
| R@10≥85% | C=256, A=2, P=4 | 87.7% | 3.5% | 4 |
| R@10≥80% | C=256, A=1, P=5 | 82.3% | 2.5% | 5 |

E5-baseと同じ構成テンプレート（C=256, assign=2-3）が有効。MiniLMの方がGapが大きいため候補効率が良い。

### プレフィックスの影響なし

MiniLMはプレフィックス不要のモデルなので、NB116で確認したquery/passageの空間差異の問題が存在しない。上記のRecallがそのまま実運用での期待値となる。

### エクスポートファイル

| 形式 | ファイル | サイズ |
|------|---------|--------|
| NumPy | `voronoi_centroids_256_minilm.npy` | 384 KB |
| JSON | `voronoi_centroids_256_minilm.json` | 2,110 KB |

E5-base (768D) の約半分のサイズ。

### 結論

1. **MiniLMはVoronoi分割と相性が良い**。Gapが大きく、少ないprobeで高Recallを達成
2. **E5-baseと同じC=256, assign=2-3のテンプレートがそのまま適用可能**
3. **プレフィックス不要**のため、実運用でのRecall低下がない（E5は1-5pp低下）
4. 英語のみの用途であれば、MiniLMの方がコスト効率が高い（モデル22M、セントロイド384KB）